# Bays (2014) Figure 2 — GP Model Predictions
## Error Distributions, Variance, Deviation from Normal, Kurtosis vs Set Size

### Intellectual Foundation

Figure 1 mapped the (λ, γ) landscape at a single location. Figure 2 fixes (λ, γ) and
varies SET SIZE using the full multi-location pipeline. This is the experiment that
most directly compares to human data.

**Panel c — Error distributions:** Broaden with set size, developing heavy tails.
**Panel d — Variance:** Supralinear power law on log-log axes (exponent > 1).
**Panel e — Deviation from circular normal:** The "Mexican hat" pattern — excess probability
at the peak AND in the tails, deficit in between. Present even at set size 1.
**Panel f — Kurtosis:** Peaks at low set sizes (where gain is highest but still finite),
then decreases as errors approach uniform.

### Key Design Choice: Full Multi-Location Factorised Decoder
Unlike Figure 1's single-location decoder, here neurons encode MULTIPLE locations
simultaneously, spikes carry entangled information, and the decoder marginalises
over non-cued items using the efficient factorised form (§5, Eq. 26).

### What to play with
- `GAMMA`: Higher gain → sharper distributions at all set sizes
- `LAMBDA_BASE`: Narrower tuning → more Fisher information → lower variance
- `SET_SIZES`: Add [1,2,3,4,5,6,7,8] for finer resolution of the power law

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time

np.random.seed(42)

# --- PARAMETERS (MODIFY THESE) ---
M = 100                    # Neurons
N_THETA = 64               # Orientation bins
N_TRIALS = 5000            # Trials per condition (10k in paper)
T_D = 0.1                  # Decoding window (s)
SIGMA_SQ = 1e-6
LAMBDA_BASE = 0.5          # GP lengthscale
GAMMA = 100.0              # Gain (Hz)
SET_SIZES = [1, 2, 4, 8]  # Items in memory
SEED = 42
N_SEEDS = 3                # Seeds for SE bands
N_BINS = 50                # Histogram bins

In [ ]:
# ============================================================
# SELF-CONTAINED CORE: GP Population + DN + Poisson + ML Decoder
# ============================================================
# Replaces imports from core.encoder.*, core.decoder.*, bays_utils

def periodic_rbf_kernel(thetas, lengthscale):
    """Periodic squared-exponential kernel."""
    diff = thetas[:, None] - thetas[None, :]
    circ = np.abs(np.angle(np.exp(1j * diff)))
    return np.exp(-0.5 * (circ / lengthscale)**2)

def sample_gp_function(K, rng):
    """Sample one function from GP(0, K)."""
    L = np.linalg.cholesky(K + 1e-6 * np.eye(len(K)))
    return L @ rng.randn(len(K))

def generate_population(M, n_theta, lengthscale, n_locations=1, seed=42):
    """Generate M neurons with GP tuning at n_locations. Returns (thetas, f_all)."""
    rng = np.random.RandomState(seed)
    thetas = np.linspace(-np.pi, np.pi, n_theta, endpoint=False)
    K = periodic_rbf_kernel(thetas, lengthscale)
    f_all = []
    for loc in range(n_locations):
        f_loc = np.zeros((M, n_theta))
        for i in range(M):
            f_loc[i] = sample_gp_function(K, rng)
        f_all.append(f_loc)
    return thetas, f_all

def dn_pointwise(r_pre, gamma, sigma_sq):
    """DN: r^post_i = γ · r^pre_i / (σ² + mean(r^pre))."""
    D = sigma_sq + np.mean(r_pre)
    return gamma * r_pre / D

def generate_spikes(rates, T_d, rng):
    """Poisson spike counts."""
    return rng.poisson(np.maximum(rates, 0) * T_d)

def compute_log_likelihood(counts, g, T_d):
    """Full Poisson ML: LL(θ) = Σ_i [n_i·log g_i(θ) - g_i(θ)·T_d]."""
    log_g = np.log(np.maximum(g, 1e-30))
    return counts @ log_g - T_d * np.sum(g, axis=0)

def compute_circular_error(theta_true, theta_hat):
    """Signed circular error in [-π, π)."""
    return np.angle(np.exp(1j * (theta_hat - theta_true)))

def circular_variance(errors):
    """Circular variance: 1 - |mean(exp(iε))|."""
    return 1.0 - np.abs(np.mean(np.exp(1j * errors)))

def circular_kurtosis(errors):
    """Circular kurtosis: κ₂/V² where κ₂ = 1-|ρ₂|, V = circ var."""
    V = circular_variance(errors)
    rho2 = np.abs(np.mean(np.exp(2j * errors)))
    kappa2 = 1.0 - rho2
    return kappa2 / max(V**2, 1e-15) if V > 1e-10 else 0.0

def compute_deviation_from_normal(errors, n_bins=50):
    """Deviation of empirical dist from matched circular normal."""
    from scipy.stats import vonmises
    bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    width = bin_edges[1] - bin_edges[0]
    emp, _ = np.histogram(errors, bins=bin_edges, density=True)
    V = circular_variance(errors)
    kappa_fit = max(0.01, 1.0 / V - 1) if V > 0.01 else 100.0
    vm_pdf = vonmises.pdf(centers, kappa_fit)
    return {'bin_centers': centers, 'empirical': emp, 'normal_fit': vm_pdf,
            'deviation': emp - vm_pdf}

# Factorised multi-location decoder
from scipy.special import logsumexp

def compute_spike_weighted_log_tuning(counts, f_list):
    """L_k(θ) = Σ_i n_i · f_{i,k}(θ) for each location k."""
    return [counts @ f_k for f_k in f_list]

def compute_marginal_log_likelihood_efficient(L_list, cued_idx):
    """Efficient factorised ML: L_c(θ) + Σ_{k≠c} logsumexp(L_k)."""
    ll = L_list[cued_idx].copy()
    for k in range(len(L_list)):
        if k != cued_idx:
            ll = ll + logsumexp(L_list[k])
    return ll

print("Core model loaded (GP + DN + Poisson + ML decoder)")

---
## Experiment: Multi-Location Encode → DN → Spike → Factorised Decode

### What is occurring
For each set size N and each trial:
1. Generate tuning at N locations, sample N true orientations
2. r^pre_i = exp(Σ_k f_{i,k}(θ_k)) — pre-normalised response
3. DN: r^post = γ · r^pre / (σ² + mean(r^pre))
4. Poisson spikes from r^post
5. Factorised ML: L_k(θ) = Σ_i n_i · f_{i,k}(θ), then marginalise non-cued

### Why marginalisation matters
Without marginalisation, the decoder would need to jointly search over all N
orientations — O(n_θ^N) complexity. The factorised form reduces this to O(N · n_θ)
by exploiting the additive structure of the log-likelihood.

In [ ]:
# ============================================================
# MAIN EXPERIMENT
# ============================================================
t0 = time.time()
max_locs = max(SET_SIZES)

all_results = {}
for s in range(N_SEEDS):
    cseed = SEED + s * 1000
    thetas, f_all = generate_population(M, N_THETA, LAMBDA_BASE, max_locs, cseed)

    for N in SET_SIZES:
        f_active = [f_all[loc] for loc in range(N)]
        rng = np.random.RandomState(cseed + N)
        errors = np.empty(N_TRIALS)

        for t in range(N_TRIALS):
            theta_idx = rng.randint(N_THETA, size=N)
            log_r_pre = np.zeros(M)
            for k in range(N):
                log_r_pre += f_active[k][:, theta_idx[k]]
            r_pre = np.exp(log_r_pre)
            rates = dn_pointwise(r_pre, GAMMA, SIGMA_SQ)
            counts = generate_spikes(rates, T_D, rng)

            L_list = compute_spike_weighted_log_tuning(counts, f_active)
            ll_marg = compute_marginal_log_likelihood_efficient(L_list, 0)
            idx_hat = np.argmax(ll_marg)
            errors[t] = compute_circular_error(thetas[theta_idx[0]], thetas[idx_hat])

        key = (s, N)
        all_results[key] = {
            'errors': errors,
            'variance': circular_variance(errors),
            'kurtosis': circular_kurtosis(errors),
            'deviation': compute_deviation_from_normal(errors, N_BINS),
        }
        print(f"  seed={s} N={N}: var={all_results[key]['variance']:.4f} "
              f"kurt={all_results[key]['kurtosis']:.2f}")

# Aggregate across seeds
summary = {}
for N in SET_SIZES:
    vs = [all_results[(s,N)]['variance'] for s in range(N_SEEDS)]
    ks = [all_results[(s,N)]['kurtosis'] for s in range(N_SEEDS)]
    emps = np.array([all_results[(s,N)]['deviation']['empirical'] for s in range(N_SEEDS)])
    devs = np.array([all_results[(s,N)]['deviation']['deviation'] for s in range(N_SEEDS)])
    summary[N] = {
        'var_mean': np.mean(vs), 'var_se': np.std(vs, ddof=1)/np.sqrt(N_SEEDS) if N_SEEDS>1 else 0,
        'kurt_mean': np.mean(ks), 'kurt_se': np.std(ks, ddof=1)/np.sqrt(N_SEEDS) if N_SEEDS>1 else 0,
        'emp_mean': np.mean(emps, axis=0), 'dev_mean': np.mean(devs, axis=0),
        'dev_se': np.std(devs, axis=0, ddof=1)/np.sqrt(N_SEEDS) if N_SEEDS>1 else np.zeros(N_BINS),
    }
bins = all_results[(0, SET_SIZES[0])]['deviation']['bin_centers']
print(f"\nDone in {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# FIGURE: 2-row × (N_ss+1) column layout
# ============================================================
n_ss = len(SET_SIZES)
RED = '#CC2222'

fig = plt.figure(figsize=(15, 7))
gs = gridspec.GridSpec(2, n_ss+1, width_ratios=[1]*n_ss + [1.3], hspace=0.4, wspace=0.35)

# Row 1: Error distributions + Variance
for i, N in enumerate(SET_SIZES):
    ax = fig.add_subplot(gs[0, i])
    ax.plot(bins, summary[N]['emp_mean'], color=RED, linewidth=1.5)
    ax.set_xlim(-np.pi, np.pi); ax.set_title(f'{N}', fontsize=13, fontweight='bold')
    ax.set_xticks([-np.pi, 0, np.pi]); ax.set_xticklabels([r'$-\pi$', '0', r'$\pi$'])
    if i == 0: ax.set_ylabel('probability density')

ax_var = fig.add_subplot(gs[0, n_ss])
ns = np.array(SET_SIZES, dtype=float)
v_mean = [summary[N]['var_mean'] for N in SET_SIZES]
ax_var.plot(ns, v_mean, 'o-', color=RED, lw=2, ms=6)
ax_var.set_xscale('log', base=2); ax_var.set_yscale('log', base=2)
ax_var.set_xticks(SET_SIZES); ax_var.set_xticklabels([str(n) for n in SET_SIZES])
ax_var.set_xlabel('items'); ax_var.set_ylabel(r'variance ($\sigma^2$)')

# Row 2: Deviation + Kurtosis
for i, N in enumerate(SET_SIZES):
    ax = fig.add_subplot(gs[1, i])
    dev = summary[N]['dev_mean']; dev_se = summary[N]['dev_se']
    ax.plot(bins, dev, color=RED, linewidth=1.5)
    ax.fill_between(bins, dev-dev_se, dev+dev_se, color=RED, alpha=0.12)
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xlim(-np.pi, np.pi); ax.set_title(f'{N}', fontsize=13, fontweight='bold')
    ax.set_xticks([-np.pi, 0, np.pi]); ax.set_xticklabels([r'$-\pi$', '0', r'$\pi$'])
    if i == 0: ax.set_ylabel(r'$\Delta$ prob density')

ax_k = fig.add_subplot(gs[1, n_ss])
k_mean = [summary[N]['kurt_mean'] for N in SET_SIZES]
ax_k.plot(ns, k_mean, 'o-', color=RED, lw=2, ms=6)
ax_k.set_xscale('log', base=2); ax_k.set_xticks(SET_SIZES)
ax_k.set_xticklabels([str(n) for n in SET_SIZES])
ax_k.set_xlabel('items'); ax_k.set_ylabel('kurtosis')

fig.suptitle('GP Population Coding — Bays (2014) Fig 2 Model Predictions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()